In [1]:
# Data Source
# Wharton Research Data Services (WRDS)
# Access date: 2026-04-20
# Period: 2010-2023
!pip freeze > requirements.txt

In [2]:
import wrds
import pandas as pd

In [3]:
def calculate_ratio(numerator, denominator, ratio_name):
    """Helper function to calculate ratio and handle division by zero."""
    if denominator == 0:
        print(f"{ratio_name}: N/A (Denominator is zero)")
        return None
    ratio = numerator / denominator
    print(f"{ratio_name}: {ratio:.4f} ({ratio*100:.2f}%)")
    return ratio

In [4]:
# Step 1: Interactive User Input
ticker_input = input("Please enter the company ticker symbol (e.g., AAPL): ").strip().upper()
year_input = input("Please enter the year (e.g., 2023): ").strip()

start_date = f"{year_input}-01-01"
end_date = f"{year_input}-12-31"

print(f"\n--- Analyzing {ticker_input} for the year {year_input} ---")

Please enter the company ticker symbol (e.g., AAPL):  AAPL
Please enter the year (e.g., 2023):  2023



--- Analyzing AAPL for the year 2023 ---


In [5]:
# Step 2: Connect to WRDS
try:
    db = wrds.Connection()
except Exception as e:
    print(f"Connection failed: {e}")
    exit()

Enter your WRDS username [15751]: zmhzmh
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\15751\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [6]:
# Step 3: Define SQL Query with additional fields
# Added: 'teq' (Total Equity), 'act' (Current Assets), 'lct' (Current Liabilities), 'invt' (Inventory)
sql_query = f"""
SELECT 
    gvkey, 
    datadate, 
    tic,
    ni AS net_profit,          -- Net Income
    at AS total_asset,         -- Total Assets
    teq AS total_equity,       -- Total Equity
    act AS current_assets,     -- Current Assets
    lct AS current_liabilities,-- Current Liabilities
    invt AS inventory          -- Inventory
FROM 
    comp.funda
WHERE 
    tic = '{ticker_input}' 
    AND datafmt = 'STD' 
    AND indfmt = 'INDL' 
    AND popsrc = 'D' 
    AND datadate >= '{start_date}'
    AND datadate <= '{end_date}'
ORDER BY 
    datadate DESC
LIMIT 1
"""

In [7]:
# Step 4: Execute Query and Calculate Ratios
print("Executing query...")
df = db.raw_sql(sql_query, coerce_float=True)

if not df.empty:
    # Extract data from the first row
    data = df.iloc[0]
        
    print(f"\n--- Financial Data for {data['tic']} ({data['datadate']}) ---")
    print(f"Net Profit: {data['net_profit']:,.2f}")
    print(f"Total Assets: {data['total_asset']:,.2f}")
    print(f"Total Equity: {data['total_equity']:,.2f}")
    print(f"Current Assets: {data['current_assets']:,.2f}")
    print(f"Current Liabilities: {data['current_liabilities']:,.2f}")
    print(f"Inventory: {data['inventory']:,.2f}")

    print("\n--- Calculated Ratios ---")
        
    # 1. ROA = Net Income / Total Assets
    calculate_ratio(data['net_profit'], data['total_asset'], "ROA")

    # 2. ROE = Net Income / Total Equity
    calculate_ratio(data['net_profit'], data['total_equity'], "ROE")

    # 3. Current Ratio = Current Assets / Current Liabilities
    calculate_ratio(data['current_assets'], data['current_liabilities'], "Current Ratio")

    # 4. Quick Ratio = (Current Assets - Inventory) / Current Liabilities
    quick_assets = data['current_assets'] - data['inventory']
    calculate_ratio(quick_assets, data['current_liabilities'], "Quick Ratio")

else:
    print(f"\nNo data found for ticker '{ticker_input}' in year '{year_input}'.")

Executing query...

--- Financial Data for AAPL (2023-09-30) ---
Net Profit: 96,995.00
Total Assets: 352,583.00
Total Equity: 62,146.00
Current Assets: 143,566.00
Current Liabilities: 145,308.00
Inventory: 6,331.00

--- Calculated Ratios ---
ROA: 0.2751 (27.51%)
ROE: 1.5608 (156.08%)
Current Ratio: 0.9880 (98.80%)
Quick Ratio: 0.9444 (94.44%)


In [ ]:
# Step 5: Financial Indicator Evaluation Function

# 1. ROA Return on Assets
def evaluate_roa(roa):
    print("\n===== ROA (Return on Assets) comment =====")
    if roa > 0.08:
        print("excellent：>8%, High asset utilization efficiency and sound overall profit quality.")
    elif 0.03 <= roa <= 0.08:
        print("average：3%~8%, Stable asset returns, staying at the industrial average level.")
    else:
        print("poor：<3%, Bulky and underutilized assets, weak profitability or even net losses.")

# 2. ROE Return on Equity
def evaluate_roe(roe):
    print("\n===== ROE (Return on Equity) comment =====")
    if roe >= 0.15:
        print("excellent：≥15%, High shareholder returns and strong competitive moat in profitability.")
    elif 0.08 <= roe < 0.15:
        print("average：8%~15%, Moderate returns, in line with the average market payoff level.")
    else:
        print("poor：<8%, Weak profit-generating capacity of internal capital and low investment value.")

# 3. Current Ratio
def evaluate_current_ratio(cr):
    print("\n===== Current Ratio comment =====")
    if cr >= 2.0:
        print("excellent：≥2.0, Sufficient current assets with extremely low short-term default risks.")
    elif 1.2 <= cr < 2.0:
        print("average：1.2~2.0, Healthy debt structure and manageable financial risks.")
    else:
        print("poor：<1.2, Heavy short-term debt pressure and high risks of capital chain tension.")

# 4. Quick Ratio
def evaluate_quick_ratio(qr):
    print("\n===== Quick Ratio comment =====")
    if qr >= 1.0:
        print("excellent：≥1.0, Able to repay short-term debts without inventory liquidation, ensuring stable cash flow.")
    elif 0.6 <= qr < 1.0:
        print("average：0.6~1.0, Basically controllable short-term solvency.")
    else:
        print("poor：<0.6, Severe short-term debt pressure and high risk of capital chain rupture.")

# Enter the company data you have calculated here.
roa = float(input("Please enter the calculation result of ROA:"))
roe = float(input("Please enter the calculation result of ROE:"))
current_ratio = float(input("Please enter the calculation result of current_ratio:"))
quick_ratio = float(input("Please enter the calculation result of quick_ratio:"))

# Run all evaluations with one click.
evaluate_roa(roa)
evaluate_roe(roe)
evaluate_current_ratio(current_ratio)
evaluate_quick_ratio(quick_ratio)

In [ ]:
pip install matplotlib pandas wrds

In [ ]:
# Step 6: Line Chart of Financial Ratios for the Past 10 Years
import matplotlib.pyplot as plt
import datetime

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Get the current year and calculate the past 10 years
current_year = int(year_input)
start_year_10y = current_year - 9
end_year_10y = current_year

print(f"\n=== Acquiring {start_year_10y} - {end_year_10y} Data of the past 10 years ===")

# SQL queries for the past 10 years
sql_10y = f"""
SELECT 
    EXTRACT(YEAR FROM datadate) AS year,
    ni / at AS roa,
    ni / teq AS roe,
    act / lct AS current_ratio,
    (act - invt) / lct AS quick_ratio
FROM comp.funda
WHERE tic = '{ticker_input}'
    AND datafmt = 'STD'
    AND indfmt = 'INDL'
    AND popsrc = 'D'
    AND datadate >= '{start_year_10y}-01-01'
    AND datadate <= '{end_year_10y}-12-31'
ORDER BY year ASC
"""

df_10y = db.raw_sql(sql_10y, coerce_float=True)

if not df_10y.empty:
    # Clean outliers
    df_10y = df_10y.replace([float('inf'), -float('inf')], float('nan')).dropna()

    years = df_10y['year'].astype(int)
    roa_list = df_10y['roa']
    roe_list = df_10y['roe']
    current_list = df_10y['current_ratio']
    quick_list = df_10y['quick_ratio']

    # Create a 2×2 subplot
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{ticker_input} Financial Ratio Trends Over the Past Decade', fontsize=16)

    # 1. ROA
    axs[0,0].plot(years, roa_list, marker='o', color='#2E86AB', linewidth=2)
    axs[0,0].set_title('ROA (Return on Assets)')
    axs[0,0].grid(True, alpha=0.3)

    # 2. ROE
    axs[0,1].plot(years, roe_list, marker='s', color='#A23B72', linewidth=2)
    axs[0,1].set_title('ROE (Return on Equity)')
    axs[0,1].grid(True, alpha=0.3)

    # 3. Current Ratio
    axs[1,0].plot(years, current_list, marker='^', color='#F18F01', linewidth=2)
    axs[1,0].set_title('Current Ratio')
    axs[1,0].grid(True, alpha=0.3)

    # 4. Quick Ratio
    axs[1,1].plot(years, quick_list, marker='d', color='#C73E1D', linewidth=2)
    axs[1,1].set_title('Quick Ratio')
    axs[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

else:
    print("Insufficient data for the past 10 years, unable to generate charts")

In [ ]:
db.close()